# QuestDB Historical Data Exploration

Query and analyze historical market data stored in QuestDB time-series database.


In [ ]:
# Install/check dependencies
required_packages = {
    'pandas': 'pandas>=2.1.0',
    'numpy': 'numpy>=1.24.0',
    'plotly': 'plotly>=5.18.0',
    'questdb': 'questdb>=1.0.0',
}

missing_packages = []
for package, requirement in required_packages.items():
    try:
        __import__(package)
        print(f"✓ {package} is installed")
    except ImportError:
        missing_packages.append(requirement)
        print(f"✗ {package} is missing")

if missing_packages:
    print(f"\n⚠ Missing packages: {', '.join(missing_packages)}")
    print("Install with: pip install -r requirements-notebooks.txt")
else:
    print("\n✓ All required packages are installed")


## Setup and Connection


In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timedelta

# Add project to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / "python"))

print(f"Project root: {project_root}")

# Import data analysis libraries
try:
    import pandas as pd
    import numpy as np
    import plotly.graph_objects as go
    print("✓ Successfully imported data analysis libraries")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("Please run the dependency check cell above and install missing packages:")
    print("  pip install -r requirements-notebooks.txt")
    raise


In [ ]:
# Import QuestDB client
try:
    from questdb.ingress import Sender
    from questdb import QuestDB
    QUESTDB_AVAILABLE = True
    print("✓ QuestDB Python client available")
except ImportError:
    QUESTDB_AVAILABLE = False
    print("⚠ QuestDB Python client not installed")
    print("  Install with: pip install questdb")

# Note: QuestDB uses HTTP REST API for queries
import requests


## Query Historical Quotes

Query QuestDB for historical quote data using SQL.


In [ ]:
# QuestDB REST API endpoint
QUESTDB_HOST = "127.0.0.1"
QUESTDB_PORT = 9000
QUESTDB_URL = f"http://{QUESTDB_HOST}:{QUESTDB_PORT}"

def query_questdb(sql: str) -> pd.DataFrame:
    """Execute SQL query against QuestDB and return DataFrame."""
    try:
        response = requests.get(
            f"{QUESTDB_URL}/exec",
            params={"query": sql},
            timeout=10
        )
        response.raise_for_status()

        # QuestDB returns CSV
        from io import StringIO
        df = pd.read_csv(StringIO(response.text))
        return df
    except Exception as e:
        print(f"Error querying QuestDB: {e}")
        return pd.DataFrame()

# Example query: Get recent quotes
symbol = "SPY"
sql = f"""
SELECT
    timestamp,
    symbol,
    bid,
    ask,
    spread,
    last
FROM quotes
WHERE symbol = '{symbol}'
    AND timestamp > now() - INTERVAL '7 days'
ORDER BY timestamp DESC
LIMIT 1000
"""

print(f"Querying QuestDB for {symbol} quotes...")
df_quotes = query_questdb(sql)

if not df_quotes.empty:
    print(f"✓ Retrieved {len(df_quotes)} quotes")
    print(df_quotes.head())
else:
    print("⚠ No data returned (QuestDB may not be running or no data available)")
    print("  Create sample data for visualization")


## Visualize Historical Data


In [ ]:
# Create visualization
if not df_quotes.empty and 'timestamp' in df_quotes.columns:
    # Convert timestamp to datetime
    df_quotes['timestamp'] = pd.to_datetime(df_quotes['timestamp'])
    df_quotes = df_quotes.sort_values('timestamp')

    fig = go.Figure()

    # Plot bid/ask
    if 'bid' in df_quotes.columns:
        fig.add_trace(go.Scatter(
            x=df_quotes['timestamp'],
            y=df_quotes['bid'],
            name='Bid',
            line=dict(color='green')
        ))

    if 'ask' in df_quotes.columns:
        fig.add_trace(go.Scatter(
            x=df_quotes['timestamp'],
            y=df_quotes['ask'],
            name='Ask',
            line=dict(color='red')
        ))

    if 'last' in df_quotes.columns:
        fig.add_trace(go.Scatter(
            x=df_quotes['timestamp'],
            y=df_quotes['last'],
            name='Last',
            line=dict(color='blue')
        ))

    fig.update_layout(
        title=f"{symbol} - Historical Quotes",
        xaxis_title="Time",
        yaxis_title="Price ($)",
        height=500
    )

    fig.show()
else:
    print("No data to visualize. QuestDB may need to be running and populated with data.")


## Analyze Spread Patterns


In [ ]:
# Analyze bid-ask spread patterns
if not df_quotes.empty and 'spread' in df_quotes.columns:
    spread_stats = df_quotes['spread'].describe()
    print("Spread Statistics:")
    print(spread_stats)

    # Plot spread distribution
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=df_quotes['spread'],
        nbinsx=50,
        name='Spread Distribution'
    ))

    fig.update_layout(
        title="Bid-Ask Spread Distribution",
        xaxis_title="Spread ($)",
        yaxis_title="Frequency",
        height=400
    )

    fig.show()
else:
    print("Spread data not available")
